In [ ]:
!pip install langchain_huggingface
!pip install langchain_core
!pip install hugging_face_hub

In [13]:
import re
import pandas as pd
from typing import TypedDict, NotRequired
from datasets import load_dataset
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser
from huggingface_hub import login

In [11]:
login()

In [12]:
class ThinkerState(TypedDict):
    id: int
    question: str
    true_answer: str
    thinker_response: NotRequired[str]
    thinker_cot: NotRequired[str]
    thinker_answer: NotRequired[str]
    is_correct: NotRequired[int]

In [15]:
class ThinkerOutput(BaseModel):
    thinker_cot: str = Field(
        description="Step-by-step reasoning for solving the problem"
    )
    thinker_answer: str = Field(
        description="Final numeric answer only (no explanation)"
    )


thinker_parser = PydanticOutputParser(
    pydantic_object=ThinkerOutput
)

In [59]:
thinker_llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3.2",

    temperature=0.2,
    max_new_tokens=768,
    do_sample=False,
    )
thinker_model = ChatHuggingFace(llm=thinker_llm)

In [60]:
thinker_prompt = PromptTemplate(
    template="""
You are a careful and rigorous math problem solver.

Your task is to solve the problem step by step.

STRICT RULES:
1. First, write detailed step-by-step reasoning.
2. DO NOT reveal the final answer anywhere in the reasoning.
3. The reasoning must NOT contain the final numeric answer.
4. You MUST always provide a final answer.
5. Even if you are unsure or think your reasoning may be incorrect, you MUST still give your best possible final answer.
6. The final answer must be a single number (no units, no explanation).
7. Your response is INVALID if thinker_answer is missing.

OUTPUT FORMAT (STRICT JSON):
{format_instructions}

Problem:
{question}
""",
    input_variables=["question"],
    partial_variables={
        "format_instructions": thinker_parser.get_format_instructions()
    }
)

In [61]:
def normalize(x):
    if x is None:
        return None
    return re.sub(r"[^\d\.-]", "", str(x))


def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0) if match else None


def is_invalid_answer(ans):
    if ans is None:
        return True
    ans = str(ans).strip().lower()
    return ans in ["", "thinker_answer", "missing"]


def remove_answer_from_cot(cot, answer):
    if cot and answer:
        cot = re.sub(rf"\b{re.escape(str(answer))}\b", "", cot)
    return cot.strip() if cot else cot


def extract_last_number_strong(text):
    tail = text[-200:]
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", tail)
    return nums[-1] if nums else None


def thinker_agent(state: ThinkerState):

    q = state["question"]
    gt = state["true_answer"]

    formatted_prompt = thinker_prompt.format(question=q)
    response = thinker_model.invoke(formatted_prompt)
    raw_text = response.content.strip()

    cot, answer = None, None

    try:
        json_text = extract_json(raw_text)
        if json_text:
            parsed = thinker_parser.parse(json_text)
            cot = parsed.thinker_cot
            answer = parsed.thinker_answer
    except Exception:
        pass


    candidate = extract_last_number_strong(raw_text)

    if is_invalid_answer(answer):
        match = re.search(r"####\s*([-0-9.,]+)", raw_text)
        if match:
            answer = match.group(1)

    if is_invalid_answer(answer):
        answer = candidate


    if is_invalid_answer(answer):
        answer = candidate

    if cot is None:
        cot = raw_text

    cot = remove_answer_from_cot(cot, answer)

    norm_pred = normalize(answer)
    norm_gt = normalize(gt)

    is_correct = int(norm_pred == norm_gt) if norm_pred and norm_gt else 0

    return {
        "thinker_response": raw_text,
        "thinker_cot": cot,
        "thinker_answer": answer,
        "is_correct": is_correct
    }

In [24]:
dataset = load_dataset("gsm8k", "main")

def extract_answer(example):
    ans = example["answer"]
    match = re.search(r"####\s*([-0-9.,]+)", ans)
    example["final_answer"] = match.group(1) if match else None
    return example

dataset = dataset.map(extract_answer)

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [25]:
print(dataset["train"][0]["question"])
print(dataset["train"][0]["final_answer"])

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
72


In [62]:
samples = dataset["train"].select(range(2))

In [63]:
for i, example in enumerate(samples):

    state = {
        "id": i+1,
        "question": example["question"],
        "true_answer": example["final_answer"]
    }

    result = thinker_agent(state)

    print("\n" + "="*60)
    print(f"ID: {i+1}")
    print("QUESTION:\n", example["question"])

    print("\nGROUND TRUTH:", example["final_answer"])
    print("THINKER ANSWER:", result.get("thinker_answer"))
    print("IS CORRECT:", result.get("is_correct"))

    print("\n--- COT ---")
    print(result.get("thinker_cot"))

    print("\n--- RAW OUTPUT (truncated) ---")
    print(result.get("thinker_response"))


ID: 1
QUESTION:
 Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

GROUND TRUTH: 72
THINKER ANSWER: 24
IS CORRECT: 0

--- COT ---
{
  "thinker_cot": "In April, Natalia sold clips to 48 friends. Assuming each friend bought one clip, the number of clips sold in April is 48. In May, she sold half as many clips as in April. Half of 48 is . So the number of clips sold in May is . The total clips sold in April and May is the sum of clips sold in April and May.",
  "thinker_answer": "

--- RAW OUTPUT (truncated) ---
{
  "thinker_cot": "In April, Natalia sold clips to 48 friends. Assuming each friend bought one clip, the number of clips sold in April is 48. In May, she sold half as many clips as in April. Half of 48 is 24. So the number of clips sold in May is 24. The total clips sold in April and May is the sum of clips sold in April and May.",
  "thinker_answer": "

ID: 2
QUESTION:
 We